# 01 · RMSNorm and pre-norm blocks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/01_norms_prenorm.ipynb) &nbsp; 📖 Concept deck: [Pre-norm and RMSNorm](https://nilesh-salpe.github.io/visual-ai/architecture/02-normalization/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** LayerNorm and RMSNorm from scratch, pre-norm and post-norm residual blocks, and an experiment that shows why deep post-norm stacks are hard to train.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _rms_ref(x, g, eps=1e-6):
    return x / np.sqrt((x * x).mean(-1, keepdims=True) + eps) * g

def _ln_ref(x, g, b, eps=1e-6):
    return (x - x.mean(-1, keepdims=True)) / np.sqrt(x.var(-1, keepdims=True) + eps) * g + b

## Task 1: LayerNorm (the 2017 version)

For each token (row) over its features: $\mathrm{LN}(x) = g \odot \frac{x-\mu}{\sqrt{\sigma^2+\epsilon}} + b$.
Two steps: **re-centre** (subtract the mean) and **re-scale** (divide by the standard deviation).

In [ ]:
def layer_norm(x, g, b, eps=1e-6):
    """x: [T, d]; g, b: [d]. Returns [T, d]."""
    # TODO: subtract the per-row mean, divide by sqrt(per-row variance + eps), then scale by g and shift by b
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_layer_norm(f):
    r = np.random.default_rng(1); x = r.normal(3, 2, (6, 16)); g = r.normal(1, .1, 16); b = r.normal(0, .1, 16)
    y = f(x, g, b)
    assert y.shape == x.shape, f"shape {y.shape} != {x.shape}"
    assert np.allclose(y, _ln_ref(x, g, b), atol=1e-6), "values differ: normalize over the LAST axis and use the variance inside the sqrt"

check("layer_norm", _test_layer_norm, layer_norm)

## Task 2: RMSNorm (what Llama uses)

Zhang & Sennrich (2019) drop the re-centring step and keep only re-scaling:
$$\mathrm{RMSNorm}(x) = g \odot \frac{x}{\sqrt{\tfrac{1}{d}\sum_i x_i^2 + \epsilon}}$$
No mean, no bias. The tests check two properties: the output's RMS is about 1 (when $g=1$), and scaling the input by any $c>0$ leaves the output unchanged (**re-scaling invariance**).

In [ ]:
def rms_norm(x, g, eps=1e-6):
    """x: [T, d]; g: [d]. Returns [T, d]."""
    # TODO: divide each row by its root-mean-square (sqrt(mean(x**2) + eps)), then scale by g
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_rms_norm(f):
    r = np.random.default_rng(2); x = r.normal(0.5, 3, (6, 32)); g = r.normal(1, .1, 32)
    y = f(x, g)
    assert y.shape == x.shape, f"shape {y.shape} != {x.shape}"
    assert np.allclose(y, _rms_ref(x, g), atol=1e-6), "values differ: use sqrt(mean(x**2) + eps) per row, and do NOT subtract the mean"
    one = f(x, np.ones(32))
    assert np.allclose(np.sqrt((one ** 2).mean(-1)), 1, atol=1e-4), "with g = 1 every output row should have RMS ≈ 1"
    assert np.allclose(f(7.5 * x, g), y, atol=1e-5), "RMSNorm must be invariant to scaling the input by a positive constant"
    assert not np.allclose(f(x + 10, g), f(x, g), atol=1e-2), "RMSNorm should NOT be shift-invariant (it does not subtract the mean)"

check("rms_norm", _test_rms_norm, rms_norm)

## Task 3: Pre-norm vs post-norm residual blocks

The 2017 Transformer is **post-norm**: $x_{l+1} = \mathrm{Norm}(x_l + F(x_l))$.
GPT-2, GPT-3 and Llama are **pre-norm**: $x_{l+1} = x_l + F(\mathrm{Norm}(x_l))$.
In pre-norm the residual stream is a clean sum: nothing ever rescales $x_l$ on its way to the output.

Implement both. `F` is any function [T, d] → [T, d]; `norm` is any function [T, d] → [T, d].

In [ ]:
def pre_norm_block(x, F, norm):
    # TODO: x + F(norm(x))
    raise NotImplementedError

def post_norm_block(x, F, norm):
    # TODO: norm(x + F(x))
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_pre_norm_block(f):
    r = np.random.default_rng(3); x = r.normal(size=(4, 8)); W = r.normal(size=(8, 8)) * .3
    F = lambda h: np.tanh(h @ W); nrm = lambda h: _rms_ref(h, np.ones(8))
    assert np.allclose(f(x, F, nrm), x + F(nrm(x))), "pre-norm must be x + F(norm(x))"
    post = post_norm_block_solution if f.__name__.endswith('_solution') else post_norm_block
    assert np.allclose(post(x, F, nrm), nrm(x + F(x))), "post-norm must be norm(x + F(x)); implement post_norm_block too"

check("pre_norm_block", _test_pre_norm_block, pre_norm_block)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: LayerNorm (the 2017 version)
def layer_norm_solution(x, g, b, eps=1e-6):
    mu = x.mean(-1, keepdims=True)
    var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps) * g + b

check("layer_norm (reference)", _test_layer_norm, layer_norm_solution)

In [ ]:
# @title Solution for Task 2: RMSNorm (what Llama uses)
def rms_norm_solution(x, g, eps=1e-6):
    rms = np.sqrt((x * x).mean(-1, keepdims=True) + eps)
    return x / rms * g

check("rms_norm (reference)", _test_rms_norm, rms_norm_solution)

In [ ]:
# @title Solution for Task 3: Pre-norm vs post-norm residual blocks
def pre_norm_block_solution(x, F, norm):
    return x + F(norm(x))

def post_norm_block_solution(x, F, norm):
    return norm(x + F(x))

check("pre_norm_block (reference)", _test_pre_norm_block, pre_norm_block_solution)

---
## Experiments
**Reproduce Xiong et al.'s (2020) finding at toy scale.** Build stacks of $L$ blocks (FFN sublayers, Xavier init, LayerNorm as in the paper) in both arrangements, and measure the gradient norm of the **last layer's** weights for a simple squared-error loss. The gradient is estimated by finite differences along 64 random directions, since $\mathbb{E}[(\nabla\cdot U)^2] = \|\nabla\|^2$ for Gaussian $U$.

Paper's result: for post-LN the gradient near the output **does not shrink with depth**, while for pre-LN it scales like $1/\sqrt{L}$. Large gradients at initialization combined with a large learning rate make post-LN unstable, which is why it needs a warm-up, and pre-LN does not.

These are random weights, so read the *trend* across depths, not the exact values.

In [ ]:
def grad_norm_last_layer(kind, L, T=16, d=64, n_dirs=64, seed=0):
    r = np.random.default_rng(seed + L)
    W1 = [r.normal(0, np.sqrt(2 / (5 * d)), (d, 4 * d)) for _ in range(L)]
    W2 = [r.normal(0, np.sqrt(2 / (5 * d)), (4 * d, d)) for _ in range(L)]
    x0, target = r.normal(size=(T, d)), r.normal(size=(T, d))
    ln = lambda h: layer_norm_solution(h, np.ones(d), np.zeros(d))
    def loss(W_last):
        x = x0
        for i in range(L):
            Wb = W_last if i == L - 1 else W2[i]
            F = lambda h, Wa=W1[i], Wb=Wb: np.maximum(0, h @ Wa) @ Wb
            x = pre_norm_block_solution(x, F, ln) if kind == 'pre' else post_norm_block_solution(x, F, ln)
        if kind == 'pre':
            x = ln(x)          # pre-norm models need a final norm
        return 0.5 * ((x - target) ** 2).sum()
    base, eps = loss(W2[-1]), 1e-6
    est = [((loss(W2[-1] + eps * U) - base) / eps) ** 2 for U in r.normal(size=(n_dirs,) + W2[-1].shape)]
    return np.sqrt(np.mean(est))

depths = [3, 6, 12, 24, 48]
print('depth L:                    ' + ''.join(f'{L:>8}' for L in depths))
for kind in ['post', 'pre']:
    print(f'{kind}-norm last-layer |grad|: ' + ''.join(f'{grad_norm_last_layer(kind, L):8.1f}' for L in depths))
print('1/sqrt(L) relative to L=3:    ' + ''.join(f'{np.sqrt(3 / L):8.2f}' for L in depths))

---
## Takeaways
* RMSNorm keeps LayerNorm's re-scaling and drops the re-centring: fewer operations, same quality (7–64% less running time in Zhang & Sennrich).
* Post-norm puts a norm on the main path after every block. At initialization the gradient of the layers near the output stays large however deep the model is, so it needs a learning-rate warm-up (Xiong et al., 2020).
* Pre-norm keeps an identity path from every layer to the output, and its gradients shrink like $1/\sqrt{L}$. It trains stably without warm-up, which is why GPT-2, GPT-3, Llama and almost every modern decoder use it. The price: the residual stream's scale grows with depth, so a **final norm** before the output head is required.